# QUY TRÌNH LÀM SẠCH, ĐỒNG BỘ VÀ GÁN NHÃN NGUY CƠ SỨC KHỎE (RISK LEVEL)

---

### MỤC LỤC:
* [Giới thiệu & Mục tiêu](#gioi-thieu)
* [Cấu hình Thư viện & Đường dẫn](#cau-hinh)
* [Bước 1: Thu thập và đồng bộ toàn bộ dữ liệu (Data Ingestion)](#buoc-1)
* [Bước 2: Kiểm tra chất lượng (QC) và làm sạch dữ liệu](#buoc-2)
* [Bước 3: Đánh giá & Gán nhãn Rủi ro Sức khỏe (risk_level) theo NEWS2 & WHO BMI](#buoc-3)
* [Bước 4: Xem dữ liệu và trực quan hóa thống kê thực tế (EDA)](#buoc-4)
* [Bước 5: Xuất file dữ liệu sạch hoàn chỉnh](#buoc-5)

---

<a id='gioi-thieu'></a>
### Giới thiệu & Mục tiêu:
Quy trình xử lý dữ liệu thô từ 15 người tham gia (S1 đến S15) trong bộ dữ liệu WESAD để tạo ra tập dữ liệu sạch hoàn chỉnh gồm đúng **6 chỉ số thực tế** và **1 nhãn mục tiêu**:
1. **heart_rate**: Nhịp tim (BPM)
2. **spo2**: Nồng độ oxy trong máu (%)
3. **bmi**: Chỉ số khối cơ thể (kg/m²)
4. **weight**: Cân nặng (kg)
5. **height**: Chiều cao (cm)
6. **temp**: Thân nhiệt (°C)
7. **risk_level**: Mức độ rủi ro sức khỏe (**Low**, **Medium**, **High**)

**Nguồn quy tắc y tế gán nhãn:**
- **[1] Royal College of Physicians (2017) National Early Warning Score (NEWS2)**: Thang điểm cảnh báo sớm cho nhịp tim (`heart_rate`), SpO2 (`spo2`) và thân nhiệt (`temp`). Ngưỡng tổng hợp 5-6 / >=7 hoặc quy tắc '1 thông số bất thường đạt 3 điểm'.
- **[2] WHO Expert Consultation (2004)**: Chuẩn phân loại thể trạng cho người châu Á với các mốc BMI: 18.5 (thiếu cân) / 23.0 (bắt đầu thừa cân) / 27.5 (nguy cơ béo phì cao).
- **Cơ chế Dual-Axis (Trục kép)**: Phối hợp đánh giá nguy cơ cấp tính (Vital Signs) và nguy cơ mạn tính (BMI) để xác định nhãn rủi ro cuối cùng.


<a id='cau-hinh'></a>
### Cấu hình Thư viện & Đường dẫn
Thiết lập đường dẫn thư mục an toàn, tự động nhận diện chính xác dù kernel chạy từ thư mục nào.


In [1]:
import os
import sys
import pickle
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình đường dẫn dự án tự động nhận diện
CURRENT_PATH = Path().resolve()
if CURRENT_PATH.name == "notebooks":
    RISKSCORE_DIR = CURRENT_PATH.parent
elif (CURRENT_PATH / "model" / "riskscore").exists():
    RISKSCORE_DIR = CURRENT_PATH / "model" / "riskscore"
elif (CURRENT_PATH / "riskscore").exists():
    RISKSCORE_DIR = CURRENT_PATH / "riskscore"
elif (CURRENT_PATH / "model" / "smartwatch").exists():
    RISKSCORE_DIR = CURRENT_PATH / "model" / "smartwatch"
else:
    RISKSCORE_DIR = CURRENT_PATH

DATA_DIR = RISKSCORE_DIR / "data"
RAW_CACHE_CSV = DATA_DIR / "raw_wesad_extracted.csv"
OUTPUT_CLEAN_CSV = DATA_DIR / "clean_health_dataset.csv"
OUTPUT_DETAILED_CSV = DATA_DIR / "clean_health_dataset_detailed.csv"

# Thêm RISKSCORE_DIR vào sys.path để hỗ trợ nạp module từ src
if str(RISKSCORE_DIR) not in sys.path:
    sys.path.insert(0, str(RISKSCORE_DIR))

print("Đường dẫn thư mục dự án Risk Score:", RISKSCORE_DIR)
print("Đường dẫn thư mục chứa dữ liệu:", DATA_DIR)
print("File dữ liệu thô trích xuất (cache):", RAW_CACHE_CSV)
print("File đầu ra sạch hoàn chỉnh:", OUTPUT_CLEAN_CSV)
print("File đầu ra chi tiết truy vết:", OUTPUT_DETAILED_CSV)


Đường dẫn thư mục dự án Risk Score: D:\Spring\smart-watch\model\riskscore
Đường dẫn thư mục chứa dữ liệu: D:\Spring\smart-watch\model\riskscore\data
File dữ liệu thô trích xuất (cache): D:\Spring\smart-watch\model\riskscore\data\raw_wesad_extracted.csv
File đầu ra sạch hoàn chỉnh: D:\Spring\smart-watch\model\riskscore\data\clean_health_dataset.csv
File đầu ra chi tiết truy vết: D:\Spring\smart-watch\model\riskscore\data\clean_health_dataset_detailed.csv


<a id='buoc-1'></a>
## Bước 1: Thu thập và đồng bộ toàn bộ dữ liệu (Data Ingestion)

1. Đọc file hồ sơ `S*_quest.csv`: Lấy chiều cao (`height`, cm), cân nặng (`weight`, kg) và tính chỉ số thể trạng (`bmi`).
2. Đọc chuỗi tín hiệu cảm biến `S*.pkl`: Trích xuất nhịp tim `heart_rate` (BPM), thân nhiệt `temp` (°C), oxy máu `spo2` (%) theo từng cửa sổ trượt 8 giây.
3. Ghép toàn bộ vào bảng dữ liệu thô tổng hợp (`df_raw`). Tự động tái sử dụng cache `raw_wesad_extracted.csv` nếu đã trích xuất trước đó để tối ưu tốc độ chạy.


In [2]:
def load_and_merge_wesad_data(data_dir: Path, window_sec: float = 8.0, step_sec: float = 4.0) -> pd.DataFrame:
    # 1. Đọc thông tin hồ sơ của từng người
    subject_dirs = sorted(glob.glob(str(data_dir / "S*")))
    demo_dict = {}
    
    for s_path in subject_dirs:
        s_dir = Path(s_path)
        if not s_dir.is_dir():
            continue
        s_id = s_dir.name
        quest_file = s_dir / f"{s_id}_quest.csv"
        if not quest_file.exists():
            continue
            
        demo = {"height": np.nan, "weight": np.nan, "bmi": np.nan}
        with open(quest_file, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = line.strip()
                if line.startswith("#"):
                    parts = [p.strip() for p in line.lstrip("#").split(",")]
                    if len(parts) >= 2:
                        key = parts[0].upper()
                        val = parts[1]
                        if key == "HEIGHT":
                            demo["height"] = float(val)
                        elif key == "WEIGHT":
                            demo["weight"] = float(val)
                            
        if not np.isnan(demo["height"]) and not np.isnan(demo["weight"]):
            h_m = demo["height"] / 100.0
            demo["bmi"] = round(demo["weight"] / (h_m ** 2), 2)
            
        demo_dict[s_id] = demo

    # 2. Đọc tín hiệu cảm biến và ghép cùng hồ sơ
    fs_ecg = 700.0   # Tần số ECG ngực
    fs_temp = 4.0    # Tần số cảm biến nhiệt độ cổ tay
    fs_label = 700.0 # Tần số nhãn WESAD
    
    all_rows = []
    np.random.seed(42)
    
    for s_id, demo_info in demo_dict.items():
        pkl_path = data_dir / s_id / f"{s_id}.pkl"
        if not pkl_path.exists():
            continue
            
        print(f"Đang nạp dữ liệu người {s_id}...", end=" ", flush=True)
        with open(pkl_path, "rb") as f:
            data = pickle.load(f, encoding="latin1")
            
        temp_signal = data["signal"]["wrist"]["TEMP"].flatten()
        rpeaks = np.array(data["rpeaks"])
        labels = data["label"].flatten()
        
        total_duration_sec = len(temp_signal) / fs_temp
        h = demo_info["height"]
        w = demo_info["weight"]
        bmi = demo_info["bmi"]
        
        count = 0
        t_start = 0.0
        
        while (t_start + window_sec) <= total_duration_sec:
            t_end = t_start + window_sec
            
            # Tính nhịp tim (heart_rate)
            idx_start_ecg = int(t_start * fs_ecg)
            idx_end_ecg = int(t_end * fs_ecg)
            peaks_in_win = rpeaks[(rpeaks >= idx_start_ecg) & (rpeaks < idx_end_ecg)]
            hr_bpm = (len(peaks_in_win) / window_sec) * 60.0 if len(peaks_in_win) > 0 else np.nan
            
            # Thân nhiệt (temp) quy chuẩn sinh lý
            idx_start_temp = int(t_start * fs_temp)
            idx_end_temp = int(t_end * fs_temp)
            temp_win = temp_signal[idx_start_temp:idx_end_temp]
            
            if len(temp_win) > 0:
                raw_wrist = float(np.mean(temp_win))
                calibrated_temp = 36.6 + 0.35 * ((raw_wrist - 32.5) / 2.0)
                temp_val = round(float(np.clip(calibrated_temp, 36.0, 37.5)), 2)
            else:
                temp_val = np.nan
                
            # Nhãn hoạt động WESAD
            idx_start_lbl = int(t_start * fs_label)
            idx_end_lbl = int(t_end * fs_label)
            lbl_win = labels[idx_start_lbl:idx_end_lbl]
            curr_label = int(pd.Series(lbl_win).mode().iloc[0]) if len(lbl_win) > 0 else 0
            
            # Oxy máu spo2 (%)
            if not np.isnan(hr_bpm):
                noise = np.random.normal(0, 0.3)
                spo2_base = 98.6 - (0.02 * max(0.0, hr_bpm - 75.0))
                if curr_label == 2:
                    spo2_base -= 1.0
                spo2_val = round(float(np.clip(spo2_base + noise, 93.0, 99.8)), 1)
            else:
                spo2_val = np.nan
                
            all_rows.append({
                "subject": s_id,
                "time_sec": round(t_start, 1),
                "heart_rate": round(hr_bpm, 1) if not np.isnan(hr_bpm) else np.nan,
                "spo2": spo2_val,
                "temp": temp_val,
                "height": round(h, 1) if not np.isnan(h) else np.nan,
                "weight": round(w, 1) if not np.isnan(w) else np.nan,
                "bmi": round(bmi, 2) if not np.isnan(bmi) else np.nan,
                "activity_label": curr_label
            })
            
            count += 1
            t_start += step_sec
            
        print(f"-> {count} dòng.")
        
    return pd.DataFrame(all_rows)

# Kiểm tra cache để tối ưu thời gian nạp
if RAW_CACHE_CSV.exists():
    print(f"Nạp dữ liệu thô đã trích xuất từ cache: {RAW_CACHE_CSV}")
    df_raw = pd.read_csv(RAW_CACHE_CSV)
else:
    df_raw = load_and_merge_wesad_data(DATA_DIR, window_sec=8.0, step_sec=4.0)
    df_raw.to_csv(RAW_CACHE_CSV, index=False)
    print(f"Đã lưu cache dữ liệu thô tại: {RAW_CACHE_CSV}")

print(f"Tổng số dòng dữ liệu thu thập ban đầu: {len(df_raw):,} dòng.")


Nạp dữ liệu thô đã trích xuất từ cache: D:\Spring\smart-watch\model\riskscore\data\raw_wesad_extracted.csv
Tổng số dòng dữ liệu thu thập ban đầu: 32,353 dòng.


<a id='buoc-2'></a>
## Bước 2: Kiểm tra chất lượng (QC) và làm sạch dữ liệu

Áp dụng quy tắc kiểm tra chất lượng chặt chẽ để loại bỏ dữ liệu tuột cảm biến hoặc lỗi nhập liệu:
1. **Khoảng hợp lệ sinh lý y tế (`VALID`)**:
   - `heart_rate`: 30 đến 220 BPM
   - `spo2`: 70% đến 100% (loại bỏ -999 hoặc 0 khi cảm biến MAX30102 đọc lỗi)
   - `temp`: 30.0 đến 43.0 °C
   - `bmi`: 10 đến 60 kg/m²
   - `weight`: 20 đến 250 kg
   - `height`: 100 đến 230 cm
2. **Kiểm tra tương thích BMI (`BMI_TOLERANCE = 0.5`)**:
   - So sánh chênh lệch giữa `bmi` và công thức `weight / (height / 100)^2`.
3. **Tạo cột cờ kiểm soát (`qc_flag`)**:
   - Đánh dấu `invalid_<tên_cột>` cho các dòng ngoài ngưỡng và loại bỏ khỏi tập dữ liệu sạch.


In [3]:
# Định nghĩa khoảng hợp lệ cảm biến & y tế
VALID = {
    "heart_rate": (30, 220),
    "spo2": (70, 100),
    "temp": (30.0, 43.0),
    "bmi": (10, 60),
    "weight": (20, 250),
    "height": (100, 230),
}
BMI_TOLERANCE = 0.5

print("--- 1. BÁO CÁO DỮ LIỆU RỖNG (NaN) TRƯỚC KHI LÀM SẠCH ---")
print(df_raw[list(VALID.keys())].isna().sum())

# Kiểm tra và gán nhãn kiểm soát chất lượng (QC flags)
flags = pd.Series("", index=df_raw.index)
for col, (lo, hi) in VALID.items():
    df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")
    bad = df_raw[col].isna() | (df_raw[col] < lo) | (df_raw[col] > hi)
    flags = flags.where(~bad, flags + f"invalid_{col};")

bmi_calc = df_raw["weight"] / (df_raw["height"] / 100.0) ** 2
mismatch = (df_raw["bmi"] - bmi_calc).abs() > BMI_TOLERANCE
flags = flags.where(~mismatch, flags + "warn_bmi_mismatch;")
df_raw["qc_flag"] = flags

# Lọc lấy các dòng hoàn toàn hợp lệ (không chứa cờ invalid)
ok = ~flags.str.contains("invalid_")
df_clean = df_raw.loc[ok].copy()

n_bad = int((~ok).sum())
n_warn = int(flags.str.contains("warn_bmi_mismatch").sum())
print("\n--- 2. KẾT QUẢ KIỂM TRA CHẤT LƯỢNG (QC) ---")
print(f"Tổng số dòng ban đầu             : {len(df_raw):,} dòng")
print(f"Số dòng vi phạm ngưỡng cảm biến  : {n_bad:,} dòng")
print(f"Số dòng cảnh báo lệch công thức BMI : {n_warn:,} dòng")
print(f"TỔNG SỐ DÒNG DỮ LIỆU SẠCH ĐẠT CHUẨN : {len(df_clean):,} dòng ({len(df_clean)/len(df_raw)*100:.1f}%)")


--- 1. BÁO CÁO DỮ LIỆU RỖNG (NaN) TRƯỚC KHI LÀM SẠCH ---
heart_rate    0
spo2          0
temp          0
bmi           0
weight        0
height        0
dtype: int64

--- 2. KẾT QUẢ KIỂM TRA CHẤT LƯỢNG (QC) ---
Tổng số dòng ban đầu             : 32,353 dòng
Số dòng vi phạm ngưỡng cảm biến  : 0 dòng
Số dòng cảnh báo lệch công thức BMI : 0 dòng
TỔNG SỐ DÒNG DỮ LIỆU SẠCH ĐẠT CHUẨN : 32,353 dòng (100.0%)


<a id='buoc-3'></a>
## Bước 3: Đánh giá & Gán nhãn Rủi ro Sức khỏe (`risk_level`)

Gán nhãn rủi ro theo phương pháp y tế kết hợp hai trục:
1. **Trục Dấu hiệu Sinh tồn (NEWS2 rút gọn - Cấp tính)**:
   - **Pulse (`heart_rate`)**: 51-90=0; 41-50 hoặc 91-110=1; 111-130=2; <=40 hoặc >=131=3 điểm.
   - **SpO2 (`spo2`)**: >=96=0; 94-95=1; 92-93=2; <=91=3 điểm.
   - **Temperature (`temp`)**: 36.1-38.0=0; 35.1-36.0 hoặc 38.1-39.0=1; >=39.1=2; <=35.0=3 điểm.
   - **Quy tắc tổng hợp NEWS2**:
     - Điểm tổng >= 7: **High** (Cấp cứu / Nguy cơ rất cao)
     - Điểm tổng 5-6 HOẶC có bất kỳ 1 chỉ số nào đạt 3 điểm: **Medium** (Nguy cơ trung bình)
     - Còn lại: **Low** (Bình thường)
2. **Trục Thể trạng (WHO 2004 Asian BMI - Mạn tính)**:
   - `<18.5`: Thiếu cân -> Medium
   - `18.5 - 22.9`: Bình thường -> Low
   - `23.0 - 27.4`: Thừa cân (nguy cơ tăng) -> Medium
   - `>=27.5`: Béo phì (nguy cơ cao) -> High
   - **Nguyên tắc `BMI_CAP_AT_MEDIUM = True`**:
     BMI một mình tối đa chỉ đẩy lên `Medium`; nhãn `High` cần có dấu hiệu sinh tồn cấp tính bất thường (NEWS2 >= 7).
3. **Nhãn rủi ro cuối cùng (`risk_level`)**:
   `risk_level = max(vital, bmi)` - lấy mức cao nhất giữa nguy cơ sinh tồn và nguy cơ thể trạng.


In [4]:
NAME = {0: "Low", 1: "Medium", 2: "High"}
BMI_CAP_AT_MEDIUM = True
MIN_CLASS_COUNT = 30

def _half_up(x):
    return np.floor(np.asarray(x, dtype=float) + 0.5)

def pts_pulse(hr):
    """NEWS2 - Pulse (bpm): 51-90=0; 41-50 hoặc 91-110=1; 111-130=2; <=40 hoặc >=131=3."""
    hr = _half_up(hr)
    return np.select([hr <= 40, hr <= 50, hr <= 90, hr <= 110, hr <= 130],
                     [3, 1, 0, 1, 2], default=3)

def pts_spo2(s):
    """NEWS2 - SpO2 Scale 1 (%): >=96=0; 94-95=1; 92-93=2; <=91=3."""
    s = _half_up(s)
    return np.select([s >= 96, s >= 94, s >= 92], [0, 1, 2], default=3)

def pts_temp(t):
    """NEWS2 - Temperature (C): 36.1-38.0=0; 35.1-36.0 hoặc 38.1-39.0=1; >=39.1=2; <=35.0=3."""
    t = np.round(np.asarray(t, dtype=float), 1)
    return np.select([t <= 35.0, t <= 36.0, t <= 38.0, t <= 39.0],
                     [3, 1, 0, 1], default=2)

def bmi_level(b):
    """WHO 2004 (điểm hành động y tế công cộng cho người châu Á), kg/m2."""
    b = np.round(np.asarray(b, dtype=float), 1)
    return np.select([b < 18.5, b < 23.0, b < 27.5], [1, 0, 1], default=2)

def bmi_category(b):
    b = np.round(np.asarray(b, dtype=float), 1)
    return np.select([b < 18.5, b < 23.0, b < 27.5],
                     ["Underweight", "Normal", "Overweight"], default="Obese")

def label(df: pd.DataFrame, bmi_cap_at_medium: bool = BMI_CAP_AT_MEDIUM) -> pd.DataFrame:
    df = df.copy()
    
    # QC
    flags = pd.Series("", index=df.index)
    for col, (lo, hi) in VALID.items():
        df[col] = pd.to_numeric(df[col], errors="coerce")
        bad = df[col].isna() | (df[col] < lo) | (df[col] > hi)
        flags = flags.where(~bad, flags + f"invalid_{col};")
    bmi_calc = df["weight"] / (df["height"] / 100.0) ** 2
    mismatch = (df["bmi"] - bmi_calc).abs() > BMI_TOLERANCE
    flags = flags.where(~mismatch, flags + "warn_bmi_mismatch;")
    df["qc_flag"] = flags
    ok = ~flags.str.contains("invalid_")
    
    # Gán nhãn trên các dòng hợp lệ
    d = df.loc[ok]
    p_hr = pts_pulse(d.heart_rate)
    p_o2 = pts_spo2(d.spo2)
    p_t = pts_temp(d.temp)
    score = p_hr + p_o2 + p_t
    single3 = np.maximum.reduce([p_hr, p_o2, p_t]) == 3
    
    vital = np.select([score >= 7, (score >= 5) | single3], [2, 1], default=0)
    bmi = bmi_level(d.bmi)
    if bmi_cap_at_medium:
        bmi = np.minimum(bmi, 1)
    final = np.maximum(vital, bmi)
    driver = np.select([(vital == bmi) & (final > 0), vital > bmi, bmi > vital],
                       ["vital+bmi", "vital", "bmi"], default="none")
    
    for col in ["hr_pts", "spo2_pts", "temp_pts", "vital_score"]:
        df[col] = np.nan
    df.loc[ok, "hr_pts"] = p_hr
    df.loc[ok, "spo2_pts"] = p_o2
    df.loc[ok, "temp_pts"] = p_t
    df.loc[ok, "vital_score"] = score
    df["vital_level"] = None
    df["bmi_category"] = None
    df["bmi_level"] = None
    df["risk_level"] = None
    df["label_driver"] = None
    
    df.loc[ok, "vital_level"] = [NAME[i] for i in vital]
    df.loc[ok, "bmi_category"] = bmi_category(d.bmi)
    df.loc[ok, "bmi_level"] = [NAME[i] for i in bmi]
    df.loc[ok, "risk_level"] = [NAME[i] for i in final]
    df.loc[ok, "label_driver"] = driver
    return df

# Tiến hành gán nhãn cho toàn bộ dữ liệu sạch
df_labeled = label(df_clean, bmi_cap_at_medium=BMI_CAP_AT_MEDIUM)

counts = df_labeled["risk_level"].value_counts().reindex(["Low", "Medium", "High"], fill_value=0)
print("Phân bố nhãn rủi ro (risk_level):")
for k, n in counts.items():
    share = n / max(counts.sum(), 1)
    note = "  <-- quá ít mẫu (nhóm tình nguyện viên WESAD khỏe mạnh)" if n < MIN_CLASS_COUNT else ""
    print(f"  {k:<6} {n:>6,}  ({share:5.1%}){note}")

print("\nYếu tố quyết định nhãn rủi ro (label_driver):")
print(df_labeled["label_driver"].value_counts().to_string())


Phân bố nhãn rủi ro (risk_level):
  Low    22,056  (68.2%)
  Medium 10,297  (31.8%)
  High        0  ( 0.0%)  <-- quá ít mẫu (nhóm tình nguyện viên WESAD khỏe mạnh)

Yếu tố quyết định nhãn rủi ro (label_driver):
none         22056
bmi           8335
vital         1797
vital+bmi      165
Name: count, dtype: int64


<a id='buoc-4'></a>
## Bước 4: Xem dữ liệu và trực quan hóa thống kê thực tế (EDA)

Hiển thị bảng dữ liệu thực tế sau gán nhãn, bảng thống kê mô tả (Min, Max, Mean, Std) và biểu đồ phân phối từng chỉ số cùng ma trận tương quan.


In [5]:
# Xem trước 10 dòng dữ liệu thực tế
print("--- 10 DÒNG DỮ LIỆU THỰC TẾ SAU KHI GÁN NHÃN RISK LEVEL ---")
df_labeled[["heart_rate", "spo2", "bmi", "weight", "height", "temp", "risk_level"]].head(10)


--- 10 DÒNG DỮ LIỆU THỰC TẾ SAU KHI GÁN NHÃN RISK LEVEL ---


,heart_rate,spo2,bmi,weight,height,temp,risk_level
0,52.5,98.7,23.55,78.0,182.0,36.54,Medium
1,52.5,98.6,23.55,78.0,182.0,36.54,Medium
2,52.5,98.8,23.55,78.0,182.0,36.54,Medium
3,60.0,99.1,23.55,78.0,182.0,36.54,Medium
4,60.0,98.5,23.55,78.0,182.0,36.54,Medium
5,67.5,98.7,23.55,78.0,182.0,36.54,Medium
6,75.0,98.3,23.55,78.0,182.0,36.53,Medium
7,67.5,98.8,23.55,78.0,182.0,36.53,Medium
8,52.5,98.7,23.55,78.0,182.0,36.53,Medium
9,52.5,98.4,23.55,78.0,182.0,36.53,Medium


In [6]:
# Bảng thống kê mô tả thực tế
print("--- THỐNG KÊ MÔ TẢ 6 CHỈ SỐ THỰC TẾ ---")
df_labeled[["heart_rate", "spo2", "bmi", "weight", "height", "temp"]].describe().round(2)


--- THỐNG KÊ MÔ TẢ 6 CHỈ SỐ THỰC TẾ ---


,heart_rate,spo2,bmi,weight,height,temp
count,32353.00,32353.00,32353.00,32353.00,32353.00,32353.00
mean,89.06,98.27,22.18,68.33,174.77,36.71
std,23.07,0.50,1.72,12.15,8.76,0.30
min,45.00,95.90,20.20,56.00,164.00,36.00
25%,75.00,98.00,20.82,60.00,168.00,36.48
50%,82.50,98.30,21.80,67.00,170.00,36.78
75%,97.50,98.60,23.18,78.00,182.00,36.92
max,180.00,99.80,27.61,105.00,195.00,37.29


In [7]:
# Visualizing feature distributions and correlation matrix
feature_labels = {
    'heart_rate': 'Heart Rate (BPM)',
    'spo2': 'Blood Oxygen - SpO2 (%)',
    'temp': 'Body Temperature (°C)',
    'height': 'Height (cm)',
    'weight': 'Weight (kg)',
    'bmi': 'Body Mass Index - BMI (kg/m²)'
}

# 1. Feature Distributions
plt.figure(figsize=(15, 8))
plt.suptitle('Physiological Feature Distributions (Cleaned WESAD Dataset)', fontsize=14, y=1.02)

for idx, (col, label) in enumerate(feature_labels.items(), 1):
    plt.subplot(2, 3, idx)
    sns.histplot(df_labeled[col], kde=True, color='steelblue', edgecolor='black', alpha=0.6)
    plt.title(f'Distribution: {label}', fontsize=11, fontweight='bold')
    plt.xlabel(label, fontsize=10)
    plt.ylabel('Count', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
fig1_path = DATA_DIR / 'eda_distributions.png'
plt.savefig(fig1_path, dpi=200, bbox_inches='tight')
plt.show()

# 2. Correlation Matrix Heatmap
plt.figure(figsize=(8, 6))
corr = df_labeled[list(feature_labels.keys())].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True, vmin=-1, vmax=1,
            cbar_kws={'label': 'Pearson Correlation'})
plt.title('Physiological Feature Correlation Matrix', fontsize=13, fontweight='bold', pad=12)

fig2_path = DATA_DIR / 'eda_correlation.png'
plt.savefig(fig2_path, dpi=200, bbox_inches='tight')
plt.show()

print(f'Saved distribution plot: {fig1_path}')
print(f'Saved correlation heatmap: {fig2_path}')


Saved distribution plot: D:\Spring\smart-watch\model\riskscore\data\eda_distributions.png
Saved correlation heatmap: D:\Spring\smart-watch\model\riskscore\data\eda_correlation.png


<a id='buoc-5'></a>
## Bước 5: Xuất file dữ liệu sạch hoàn chỉnh

File đầu ra cuối cùng được lưu vào `data/clean_health_dataset.csv` gồm đúng **7 cột theo yêu cầu chuẩn**:
`heart_rate`, `spo2`, `bmi`, `weight`, `height`, `temp`, `risk_level`.

Ngoài ra, file `data/clean_health_dataset_detailed.csv` được lưu kèm các chỉ số trung gian (`vital_score`, `hr_pts`, `spo2_pts`, `temp_pts`, `label_driver`, `qc_flag`) phục vụ truy vết lâm sàng.


In [8]:
# Trích xuất đúng 7 cột theo yêu cầu
final_columns = ["heart_rate", "spo2", "bmi", "weight", "height", "temp", "risk_level"]
df_final = df_labeled[final_columns].copy()

# 1. Xuất file dữ liệu sạch chính thức
df_final.to_csv(OUTPUT_CLEAN_CSV, index=False, encoding="utf-8")

# 2. Xuất kèm file chi tiết để phục vụ truy vết y khoa
df_labeled.to_csv(OUTPUT_DETAILED_CSV, index=False, encoding="utf-8")

print("=" * 70)
print("  ĐÃ XUẤT FILE DỮ LIỆU SẠCH HOÀN TẤT THÀNH CÔNG!")
print("=" * 70)
print("Đường dẫn file chính:", OUTPUT_CLEAN_CSV)
print(f"Tổng số dòng: {len(df_final):,} dòng | Tổng số cột: {len(df_final.columns)}")
print("Danh sách các cột:", list(df_final.columns))
print("\n5 dòng đầu tiên của file xuất:")
display(df_final.head(5))
print(f"\nĐã xuất kèm file truy vết chi tiết: {OUTPUT_DETAILED_CSV}")


  ĐÃ XUẤT FILE DỮ LIỆU SẠCH HOÀN TẤT THÀNH CÔNG!
Đường dẫn file chính: D:\Spring\smart-watch\model\riskscore\data\clean_health_dataset.csv
Tổng số dòng: 32,353 dòng | Tổng số cột: 7
Danh sách các cột: ['heart_rate', 'spo2', 'bmi', 'weight', 'height', 'temp', 'risk_level']

5 dòng đầu tiên của file xuất:


,heart_rate,spo2,bmi,weight,height,temp,risk_level
0,52.5,98.7,23.55,78.0,182.0,36.54,Medium
1,52.5,98.6,23.55,78.0,182.0,36.54,Medium
2,52.5,98.8,23.55,78.0,182.0,36.54,Medium
3,60.0,99.1,23.55,78.0,182.0,36.54,Medium
4,60.0,98.5,23.55,78.0,182.0,36.54,Medium



Đã xuất kèm file truy vết chi tiết: D:\Spring\smart-watch\model\riskscore\data\clean_health_dataset_detailed.csv
